# rizzvision-v2: Multi-class Clothing Classifier (tops / bottoms / other)

**Kaggle notebook** — attach the dataset `nitin2807/rizzvision-clothing-dataset-v2` before running.

**Enable GPU:** Settings → Accelerator → P100.

**Targets:** overall accuracy ≥ 92%, false positive rate < 2% per class, minimum confidence threshold 0.70.

**Outputs** saved to `/kaggle/working/` and available in the Output tab after the run:
- `clothing_classifier_v2.keras`
- `thresholds_v2.json`

**Resuming from checkpoint:** Set `RESUME_FROM_CHECKPOINT = True`, point `CHECKPOINT_PATH` to the uploaded `.keras` file, and set `PHASE2_DONE` to the last completed Phase 2 epoch.

In [ ]:
import os, json, math
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

# ── Paths ──────────────────────────────────────────────────────────────────────
DATASET_DIR = '/kaggle/input/datasets/nitin2807/rizzvision-clothing-dataset-v2'
OUT_DIR     = '/kaggle/working'

# ── Checkpoint resume ──────────────────────────────────────────────────────────
RESUME_FROM_CHECKPOINT = False
CHECKPOINT_PATH = '/kaggle/input/datasets/nitin2807/rv-train2-checkpoint/clothing_classifier_v2.keras'
PHASE2_DONE = 0   # number of Phase 2 epochs already completed

# ── Hyperparams ────────────────────────────────────────────────────────────────
INPUT_SIZE        = 300
BATCH_SIZE        = 48       # P100 16 GB handles batch 48 comfortably
CLASSES           = ['tops', 'bottoms', 'other']
NUM_CLASSES       = len(CLASSES)
MIN_THRESHOLD     = 0.70     # floor — never go below this
TARGET_FPR        = 0.015    # 1.5% calibration target — gives ~2% headroom on test set
PHASE1_EPOCHS     = 5
PHASE2_EPOCHS     = 50
UNFREEZE_LAYERS   = 150
LABEL_SMOOTHING   = 0.1
PATIENCE          = 8

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

for split in ['train', 'val', 'test']:
    path = os.path.join(DATASET_DIR, split)
    if os.path.isdir(path):
        counts = {c: len(os.listdir(os.path.join(path, c)))
                  for c in CLASSES if os.path.isdir(os.path.join(path, c))}
        print(f'{split}: {counts}')
    else:
        print(f'WARNING: {path} not found — attach the dataset in Settings → Add data')

In [ ]:
# ── Keras deserialization patch ────────────────────────────────────────────────
# Strips the unknown `quantization_config` field from Dense configs so that
# checkpoints saved with newer Keras load correctly on older Kaggle kernels.
from keras.src.saving import serialization_lib

_orig_deserialize = serialization_lib.deserialize_keras_object

def _patched_deserialize(config, *args, **kwargs):
    if (isinstance(config, dict)
            and config.get('class_name') == 'Dense'
            and isinstance(config.get('config'), dict)):
        config['config'].pop('quantization_config', None)
    return _orig_deserialize(config, *args, **kwargs)

serialization_lib.deserialize_keras_object = _patched_deserialize
print('Deserialization patch applied.')

In [ ]:
# ── Dataset pipeline ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 40.0)
    image = tf.image.random_contrast(image, 0.75, 1.25)
    image = tf.image.random_saturation(image, 0.75, 1.25)
    image = tf.image.random_crop(image, [int(INPUT_SIZE * 0.9), int(INPUT_SIZE * 0.9), 3])
    image = tf.image.resize(image, [INPUT_SIZE, INPUT_SIZE])
    image = tf.clip_by_value(image, 0, 255)
    return image, label

def make_dataset(split, augment_fn=None, shuffle=False):
    ds = keras.utils.image_dataset_from_directory(
        os.path.join(DATASET_DIR, split),
        class_names=CLASSES,
        image_size=(INPUT_SIZE, INPUT_SIZE),
        batch_size=None,          # unbatched so augment receives (H, W, 3)
        label_mode='categorical',
        shuffle=shuffle,
        seed=42,
    )
    if augment_fn:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_dataset('train', augment_fn=augment, shuffle=True)
val_ds   = make_dataset('val')
test_ds  = make_dataset('test')
print('Datasets loaded. Class order:', CLASSES)

In [ ]:
# ── Build or load model ───────────────────────────────────────────────────────
CKPT_PATH = os.path.join(OUT_DIR, 'clothing_classifier_v2.keras')

def build_model(trainable_base=False):
    base = keras.applications.EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=(INPUT_SIZE, INPUT_SIZE, 3),
    )
    base.trainable = trainable_base
    inputs  = keras.Input(shape=(INPUT_SIZE, INPUT_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return keras.Model(inputs, outputs), base

if RESUME_FROM_CHECKPOINT:
    print(f'Loading checkpoint from {CHECKPOINT_PATH} ...')
    model = keras.models.load_model(CHECKPOINT_PATH)
    base  = next(l for l in model.layers if 'efficientnet' in l.name.lower())
    print('Checkpoint loaded. Skipping Phase 1.')
else:
    model, base = build_model(trainable_base=False)

model.summary()

In [ ]:
# ── Phase 1: train head only (skipped when resuming) ─────────────────────────
if not RESUME_FROM_CHECKPOINT:
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=['accuracy', keras.metrics.AUC(name='auc', multi_label=False)],
    )
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=PHASE1_EPOCHS,
        callbacks=[
            keras.callbacks.ModelCheckpoint(
                CKPT_PATH,
                monitor='val_accuracy', mode='max',
                save_best_only=True, verbose=1,
            ),
        ],
    )
    print('Phase 1 complete.')
else:
    print(f'Phase 1 skipped (resuming at Phase 2 epoch {PHASE2_DONE + 1}).')

In [ ]:
# ── Phase 2: fine-tune top layers ────────────────────────────────────────────
REMAINING = PHASE2_EPOCHS - PHASE2_DONE

base.trainable = True
for layer in base.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

trainable = sum(1 for l in base.layers if l.trainable)
print(f'Unfreezing top {trainable} of {len(base.layers)} base layers')

total_steps = PHASE2_EPOCHS * len(train_ds)
steps_done  = PHASE2_DONE   * len(train_ds)
initial_lr  = 1e-6 + 0.5 * (1 + math.cos(math.pi * steps_done / total_steps)) * (1e-4 - 1e-6)

print(f'Phase 2: running {REMAINING} of {PHASE2_EPOCHS} epochs (starting LR ≈ {initial_lr:.2e})')

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=initial_lr,
    decay_steps=REMAINING * len(train_ds),
    alpha=1e-6,
)

model.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy', keras.metrics.AUC(name='auc', multi_label=False)],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=REMAINING,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            CKPT_PATH,
            monitor='val_accuracy', mode='max',
            save_best_only=True, verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=PATIENCE, mode='max',
            restore_best_weights=True, verbose=1,
        ),
    ],
)

print('Phase 2 complete.')

In [ ]:
# ── Per-class threshold calibration ───────────────────────────────────────────
# Finds the lowest threshold per class that keeps FPR < TARGET_FPR (2%),
# with MIN_THRESHOLD (0.70) as the hard floor.
# FPR here = false positives / total negatives for that class.
best_model = keras.models.load_model(CKPT_PATH)

val_probs, val_labels = [], []
for batch_x, batch_y in val_ds:
    val_probs.extend(best_model.predict(batch_x, verbose=0).tolist())
    val_labels.extend(np.argmax(batch_y.numpy(), axis=1).tolist())

val_probs  = np.array(val_probs)   # (N, 3)
val_labels = np.array(val_labels)  # (N,)

thresholds = {}
print(f'Calibrating thresholds (floor={MIN_THRESHOLD}, target FPR<{TARGET_FPR*100:.0f}%)')
print()

for i, cls in enumerate(CLASSES):
    class_probs = val_probs[:, i]
    negatives   = (val_labels != i).sum()   # total true negatives for this class

    chosen = MIN_THRESHOLD  # start at floor
    for thresh in np.arange(MIN_THRESHOLD, 0.99, 0.01):
        accepted = class_probs >= thresh
        fp  = ((accepted) & (val_labels != i)).sum()
        fpr = fp / (negatives + 1e-9)
        if fpr <= TARGET_FPR:
            chosen = float(thresh)
            break

    # Report stats at chosen threshold
    accepted  = class_probs >= chosen
    tp  = ((accepted) & (val_labels == i)).sum()
    fp  = ((accepted) & (val_labels != i)).sum()
    fpr = fp / (negatives + 1e-9)
    precision = tp / (tp + fp + 1e-9)

    thresholds[cls] = round(chosen, 3)
    print(f'{cls}: threshold={chosen:.2f}  FPR={fpr*100:.2f}%  precision={precision:.3f}  accepted={accepted.sum()}')

thresholds_out = {'thresholds': thresholds, 'classes': CLASSES}
thresholds_path = os.path.join(OUT_DIR, 'thresholds_v2.json')
with open(thresholds_path, 'w') as f:
    json.dump(thresholds_out, f, indent=2)

print()
print('thresholds_v2.json:', thresholds_out)

In [ ]:
# ── Test set evaluation ───────────────────────────────────────────────────────
MAX_FPR = 0.02   # user's actual target — separate from calibration TARGET_FPR

test_probs, test_labels = [], []
for batch_x, batch_y in test_ds:
    test_probs.extend(best_model.predict(batch_x, verbose=0).tolist())
    test_labels.extend(np.argmax(batch_y.numpy(), axis=1).tolist())

test_probs  = np.array(test_probs)
test_labels = np.array(test_labels)
test_preds  = test_probs.argmax(axis=1)
test_max    = test_probs.max(axis=1)

overall_accuracy = (test_preds == test_labels).mean()

print(f'\n=== Test Set Results ===')
print(f'Overall Accuracy : {overall_accuracy:.4f}  (target >= 0.92)')
print()

# Per-class FPR at calibrated thresholds
print('Per-class FPR at calibrated thresholds:')
all_fpr_ok = True
for i, cls in enumerate(CLASSES):
    thresh    = thresholds[cls]
    accepted  = test_probs[:, i] >= thresh
    negatives = (test_labels != i).sum()
    fp        = ((accepted) & (test_labels != i)).sum()
    tp        = ((accepted) & (test_labels == i)).sum()
    fpr       = fp / (negatives + 1e-9)
    precision = tp / (tp + fp + 1e-9)
    status    = 'OK' if fpr <= MAX_FPR else 'ABOVE TARGET'
    if fpr > MAX_FPR:
        all_fpr_ok = False
    print(f'  {cls}: threshold={thresh:.2f}  FPR={fpr*100:.2f}%  precision={precision:.3f}  [{status}]')

accepted_any = test_max >= min(thresholds.values())
acc_accepted = (test_preds[accepted_any] == test_labels[accepted_any]).mean() \
               if accepted_any.any() else 0.0

print()
print(f'Accept rate        : {accepted_any.mean()*100:.1f}%')
print(f'Accuracy (accepted): {acc_accepted:.4f}')
print()
print('Targets met:', 'YES ✓' if overall_accuracy >= 0.92 and all_fpr_ok else 'NO — see above')
print()
print('Per-class report:')
print(classification_report(test_labels, test_preds, target_names=CLASSES))
print('Confusion matrix (rows=actual, cols=predicted):')
print(confusion_matrix(test_labels, test_preds))
print()
print('Output files:')
for fname in ['clothing_classifier_v2.keras', 'thresholds_v2.json']:
    path = os.path.join(OUT_DIR, fname)
    size = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    print(f'  {fname}: {size:.1f} MB')
print('\nDownload from the Output tab on the right →')